# ULTIMATIVE KAGGLE COMPETITION 🗣️

### Importing Libraries

In [1]:
# -------------------------------------
# Standard libraries
# -------------------------------------
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# -------------------------------------
# Scikit-learn
# -------------------------------------
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score
from sklearn.datasets import make_regression
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression, mutual_info_regression, RFECV, SelectFromModel
from sklearn.base import clone
from sklearn import set_config

# -------------------------------------
# Config
# -------------------------------------
set_config(transform_output='pandas')

### Loading Data & Train-Test Split

In [2]:
# -------------------------------------
# Reading Data
# -------------------------------------
url1 = 'https://drive.google.com/file/d/11uf0ZqHzD0dgcNoO3_z2ejDtJxr7pq_G/view?usp=sharing'
path1 = 'https://drive.google.com/uc?export=download&id='+url1.split('/')[-2]
data = pd.read_csv(path1)

url2 = 'https://drive.google.com/file/d/1eO1wqRdTCaCWvO7CyIeJd9deMyPWd8Y-/view?usp=sharing'
path2 = 'https://drive.google.com/uc?export=download&id='+url2.split('/')[-2]
submission_data = pd.read_csv(path2)

# -------------------------------------
# Feature (X) and Target Separation (y)
# -------------------------------------
X = data.copy()
y = X.pop('SalePrice')

# -------------------------------------
# Train-Test Split
# -------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=3003)

### Preprocessing

In [3]:
# -------------
# Column Groups
# -------------

ordinal_num = ['OverallQual', 'OverallCond']
nominal_num = ['MSSubClass']
num_features = [col for col in X.select_dtypes(include='number').columns if col not in ordinal_num + nominal_num]

ordinal_cat = {
    'LotShape':['IR3', 'IR2', 'IR1', 'Reg'],
    'LandContour':['Low', 'HLS', 'Bnk', 'Lvl'],
    'Utilities':['ELO', 'NoSeWa', 'NoSewr', 'AllPub'],
    'LandSlope':['Sev','Mod','Gtl'],
    'ExterQual':['Po','Fa','TA','Gd','Ex'],
    'ExterCond':['Po','Fa','TA','Gd','Ex'],
    'BsmtQual':['N_A','Po','Fa','TA','Gd','Ex'],
    'BsmtCond':['N_A','Po','Fa','TA','Gd','Ex'],
    'BsmtExposure':['N_A','No','Mn','Av','Gd'],
    'BsmtFinType1':['N_A','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'BsmtFinType2':['N_A','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'HeatingQC':['Po','Fa','TA','Gd','Ex'],
    'Electrical':['N_A','Mix','FuseP','FuseF','FuseA','SBrkr'],
    'KitchenQual':['Po','Fa','TA','Gd','Ex'],
    'Functional':['Sal','Sev','Maj2','Maj1','Mod','Min2','Min1','Typ'],
    'FireplaceQu':['N_A','Po','Fa','TA','Gd','Ex'],
    'GarageFinish':['N_A','Unf','RFn','Fin'],
    'GarageQual':['N_A','Po','Fa','TA','Gd','Ex'],
    'GarageCond':['N_A','Po','Fa','TA','Gd','Ex'],
    'PavedDrive':['N','P','Y'],
    'PoolQC':['N_A','Fa','TA','Gd','Ex'],
    'Fence': ['N_A', 'MnWw', 'MnPrv', 'GdWo', 'GdPrv']
    }

binary_cols = ['CentralAir','Street']

nominal_cat = [col for col in X.select_dtypes(exclude='number').columns if col not in list(ordinal_cat.keys()) + binary_cols]

# ------------------------------------------------
# Preprocessor (Pipelines + ColumnTransformer)
# ------------------------------------------------

preprocessor = ColumnTransformer(transformers=[
    # Plain numerical: impute with mean, then scale
    ('num', make_pipeline(
        SimpleImputer(strategy='mean'),
        StandardScaler()
    ), num_features),

    # Ordinal numerical
    ('ordinal_num', make_pipeline(
        SimpleImputer(strategy='mean'),
        StandardScaler()
    ), ordinal_num),

    # Nominal numerical: impute, then one-hot encode
    ('nominal_num', make_pipeline(
        SimpleImputer(strategy='most_frequent'),
        OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    ), nominal_num),

    # Ordinal Categorical: impute with N_A, then ordinal encode
    ('ordinal_cat', make_pipeline(
        SimpleImputer(strategy='constant', fill_value='N_A'),
        OrdinalEncoder(categories=list(ordinal_cat.values()),
                       handle_unknown='use_encoded_value',
                       unknown_value=-1)
    ), list(ordinal_cat.keys())),

    # Binary categorical: impute, then ordinal encode as 0/1
    ('binary', make_pipeline(
        SimpleImputer(strategy='most_frequent'),
        OrdinalEncoder(handle_unknown='use_encoded_value',
                       unknown_value=-1)
    ), binary_cols),

    # Nominal categorical: impute with N_A, then one-hot encode
    ('nominal_cat', make_pipeline(
        SimpleImputer(strategy='constant', fill_value='N_A'),
        OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    ), nominal_cat),
])

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ordinal_num', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_n

### Baseline Model

Function that scores the models' performances

In [4]:
def score_models(feat_select_method, tree_pipe, knn_pipe, X_tr=X_train, y_tr=y_train, X_te=X_test, y_te=y_test):
  tree_pipe.fit(X_tr, y_tr)
  knn_pipe.fit(X_tr, y_tr)

  # Score their predictions
  scores = {
      'Feature Selection': feat_select_method,
      'Decision Tree': r2_score(y_te, tree_pipe.predict(X_te)),
      'KNN': r2_score(y_te, knn_pipe.predict(X_te))
  }
  return scores

Basic Pipelines without Feature Selection

In [5]:
# Creating DecisionTreeRegressor Pipeline
base_tree = make_pipeline(preprocessor,
    DecisionTreeRegressor(random_state=42)
)

# Creating KNeighborsRegressor Pipeline
base_knn = make_pipeline(preprocessor,
    KNeighborsRegressor(n_neighbors=5)
)

Saving scores in a DataFrame

In [6]:
method_scores = []
method_scores.append(score_models('Baseline', base_tree, base_knn))
pd.DataFrame(method_scores)

,Feature Selection,Decision Tree,KNN
0,Baseline,0.574021,0.530017


### Feature Selections with Variance Threshold

Before fitting

In [7]:
range_var_df = (pd.DataFrame({
    'Range': X_train.select_dtypes(include='number').max() - X_train.select_dtypes(include='number').min(),
    'Variance': X_train.select_dtypes(include='number').var()})
    .sort_values(by='Variance'))

range_var_df.head(10)

,Range,Variance
KitchenAbvGr,3.0,0.050052
BsmtHalfBath,2.0,0.057546
HalfBath,2.0,0.254358
BsmtFullBath,3.0,0.270621
FullBath,3.0,0.301469
Fireplaces,3.0,0.425021
GarageCars,4.0,0.551182
BedroomAbvGr,8.0,0.678574
OverallCond,7.0,1.211888
YrSold,4.0,1.804715


After fitting

In [8]:
# Preprocessed version of X_train & X_test
preproc_clone = clone(preprocessor)
X_train_preproc = preproc_clone.fit_transform(X_train, y_train)
X_test_preproc = preproc_clone.transform(X_test)

# Variance Threshold
range_var_df = (pd.DataFrame({
    'Range': X_train_preproc.max() - X_train_preproc.min(),
    'Variance': X_train_preproc.var()})
    .sort_values(by='Variance'))

range_var_df.head(10)

,Range,Variance
nominal_cat__Exterior1st_AsphShn,1.0,0.000856
nominal_cat__Exterior2nd_AsphShn,1.0,0.000856
nominal_cat__RoofMatl_Membran,1.0,0.000856
nominal_cat__Neighborhood_Blueste,1.0,0.000856
nominal_cat__LotConfig_FR3,1.0,0.000856
nominal_cat__Condition2_RRAn,1.0,0.000856
nominal_cat__Exterior2nd_Other,1.0,0.000856
nominal_cat__Exterior2nd_CBlock,1.0,0.000856
nominal_cat__Exterior1st_CBlock,1.0,0.000856
nominal_cat__Condition2_RRNn,1.0,0.000856


Variance Pipelines

In [9]:
# 1st Iteration
var0_02_tree = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    VarianceThreshold(threshold=0.02),
    DecisionTreeRegressor(random_state=42)
)
var0_02_knn = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    VarianceThreshold(threshold=0.02),
    KNeighborsRegressor(n_neighbors=5)
)
method_scores.append(score_models('Variance Threshold (0.02)', var0_02_tree, var0_02_knn))
pd.DataFrame(method_scores)

,Feature Selection,Decision Tree,KNN
0,Baseline,0.574021,0.530017
1,Variance Threshold (0.02),0.550792,0.614680


In [10]:
# Investigating how many features were dropped in the 1st it.
print(f"""Features before applying threshold: {var0_02_tree['variancethreshold'].n_features_in_}
Features after applying threshold: {var0_02_tree['decisiontreeregressor'].n_features_in_}""")
print(f'Difference: {(var0_02_tree['variancethreshold'].n_features_in_)- var0_02_tree['decisiontreeregressor'].n_features_in_} ')

Features before applying threshold: 229
Features after applying threshold: 119
Difference: 110 


In [11]:
# 2nd Iteration
var0_tree = make_pipeline(
    preprocessor,
    VarianceThreshold(threshold=0),
    DecisionTreeRegressor(random_state=42)
)
var0_knn = make_pipeline(
    preprocessor,
    VarianceThreshold(threshold=0),
    KNeighborsRegressor(n_neighbors=5)
)
method_scores.append(score_models('Variance Threshold (0)', var0_tree, var0_knn))
pd.DataFrame(method_scores)

,Feature Selection,Decision Tree,KNN
0,Baseline,0.574021,0.530017
1,Variance Threshold (0.02),0.550792,0.614680
2,Variance Threshold (0),0.574021,0.530017


In [12]:
# Investigating how many features were dropped in the 2nd it.
print(f"""Features before applying threshold: {var0_tree['variancethreshold'].n_features_in_}
Features after applying threshold: {var0_tree['decisiontreeregressor'].n_features_in_}""")
print(f'Difference: {(var0_tree['variancethreshold'].n_features_in_)- var0_tree['decisiontreeregressor'].n_features_in_} ')

Features before applying threshold: 229
Features after applying threshold: 229
Difference: 0 


### Feature Selection with Collinearity

In [13]:
# Set the correlation threshold to consider columns as highly correlated
correlation_threshold = 0.90

# Calculate the absolute correlation matrix for the feature matrix X_train_var2
corrMatrix = X_train_preproc.corr().abs()

# Initialise an empty list to store the pairs of highly correlated columns
highly_correlated_columns = []

# Get the number of features (columns) in the correlation matrix
num_features = len(corrMatrix.columns)

# Loop through the upper triangle of the correlation matrix to find highly correlated columns
# Note: We start from i+1 to avoid redundancy as correlation_matrix is symmetric
for i in range(num_features):
  for j in range(i + 1, num_features):
    # Check if the correlation value between columns i and j is greater than or equal to the threshold
    if corrMatrix.iloc[i, j] >= correlation_threshold:
      # Append the tuple (column_i, column_j) to the list of highly correlated columns
      highly_correlated_columns.append((corrMatrix.columns[i], corrMatrix.columns[j], f"correlation = {round(corrMatrix.iloc[i, j], 2)}"))

print('Highly correlated columns:', highly_correlated_columns)

Highly correlated columns: [('num__PoolArea', 'ordinal_cat__PoolQC', 'correlation = 0.93'), ('nominal_num__MSSubClass_50', 'nominal_cat__HouseStyle_1.5Fin', 'correlation = 0.94'), ('nominal_num__MSSubClass_80', 'nominal_cat__HouseStyle_SLvl', 'correlation = 0.95'), ('nominal_num__MSSubClass_90', 'nominal_cat__BldgType_Duplex', 'correlation = 1.0'), ('nominal_num__MSSubClass_190', 'nominal_cat__BldgType_2fmCon', 'correlation = 0.98'), ('ordinal_cat__GarageQual', 'ordinal_cat__GarageCond', 'correlation = 0.96'), ('ordinal_cat__GarageQual', 'nominal_cat__GarageType_N_A', 'correlation = 0.94'), ('ordinal_cat__GarageCond', 'nominal_cat__GarageType_N_A', 'correlation = 0.95'), ('nominal_cat__Condition2_RRAe', 'nominal_cat__MiscFeature_Gar2', 'correlation = 1.0'), ('nominal_cat__RoofStyle_Gable', 'nominal_cat__RoofStyle_Hip', 'correlation = 0.93'), ('nominal_cat__Exterior1st_AsphShn', 'nominal_cat__Exterior2nd_AsphShn', 'correlation = 1.0'), ('nominal_cat__Exterior1st_CBlock', 'nominal_cat__E

In [14]:
# Finding out which features to drop
to_drop = [element_a for element_a, element_b, element_c in highly_correlated_columns]
to_drop

['num__PoolArea',
 'nominal_num__MSSubClass_50',
 'nominal_num__MSSubClass_80',
 'nominal_num__MSSubClass_90',
 'nominal_num__MSSubClass_190',
 'ordinal_cat__GarageQual',
 'ordinal_cat__GarageQual',
 'ordinal_cat__GarageCond',
 'nominal_cat__Condition2_RRAe',
 'nominal_cat__RoofStyle_Gable',
 'nominal_cat__Exterior1st_AsphShn',
 'nominal_cat__Exterior1st_CBlock',
 'nominal_cat__Exterior1st_CemntBd',
 'nominal_cat__Exterior1st_MetalSd',
 'nominal_cat__Exterior1st_VinylSd',
 'nominal_cat__MiscFeature_N_A',
 'nominal_cat__SaleType_New']

In [15]:
# Drop the columns from the train set
X_train_corr = X_train_preproc.drop(columns=to_drop).copy()

# Drop the columns from the test set
X_test_corr = X_test_preproc.drop(columns=to_drop).copy()

In [16]:
corr_tree = make_pipeline(
    DecisionTreeRegressor(random_state=42)
)
corr_knn = make_pipeline(
    KNeighborsRegressor(n_neighbors=5)
)

method_scores.append(score_models('Collinearity Threshold (0.90)', corr_tree, corr_knn, X_train_corr, y_train, X_test_corr, y_test))
pd.DataFrame(method_scores)

,Feature Selection,Decision Tree,KNN
0,Baseline,0.574021,0.530017
1,Variance Threshold (0.02),0.550792,0.614680
2,Variance Threshold (0),0.574021,0.530017
3,Collinearity Threshold (0.90),0.547674,0.505228


### Feature Selection with SelectKBest

In [17]:
kbest_tree = make_pipeline(
    preprocessor,
    SelectKBest(score_func=f_regression, k=10),
    DecisionTreeRegressor(random_state=42)
)
kbest_knn = make_pipeline(
    preprocessor,
    SelectKBest(score_func=f_regression, k=10),
    KNeighborsRegressor(n_neighbors=5)
)

method_scores.append(score_models('SelectKBest (k=10)', kbest_tree, kbest_knn))
pd.DataFrame(method_scores)

,Feature Selection,Decision Tree,KNN
0,Baseline,0.574021,0.530017
1,Variance Threshold (0.02),0.550792,0.614680
2,Variance Threshold (0),0.574021,0.530017
3,Collinearity Threshold (0.90),0.547674,0.505228
4,SelectKBest (k=10),0.339865,0.619578


In [18]:
sfm_tree = make_pipeline(
    preprocessor,
    SelectFromModel(DecisionTreeRegressor(), threshold=None),
    DecisionTreeRegressor(random_state=42)
)
sfm_knn = make_pipeline(
    preprocessor,
    SelectFromModel(DecisionTreeRegressor(), threshold=None),
    KNeighborsRegressor(n_neighbors=5)
)

method_scores.append(score_models('SelectFromModel', sfm_tree, sfm_knn))
pd.DataFrame(method_scores)

,Feature Selection,Decision Tree,KNN
0,Baseline,0.574021,0.530017
1,Variance Threshold (0.02),0.550792,0.614680
2,Variance Threshold (0),0.574021,0.530017
3,Collinearity Threshold (0.90),0.547674,0.505228
4,SelectKBest (k=10),0.339865,0.619578
5,SelectFromModel,0.497098,0.505258


### Hyperparameter Tuning with `GridSearchCV`

In [19]:
pipe = make_pipeline(
    preprocessor,
    SelectKBest(score_func=f_regression),
    DecisionTreeRegressor(random_state=42)
)

param_grid = {
    'selectkbest__k': [5, 10, 20, 30],
    'decisiontreeregressor__max_depth': [3, 5, 10, None],
    'decisiontreeregressor__min_samples_split': [2, 5, 10],
}

grid_search = GridSearchCV(pipe, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

Best params: {'decisiontreeregressor__max_depth': None, 'decisiontreeregressor__min_samples_split': 10, 'selectkbest__k': 20}
Best score: 0.7556829728884782


### Final evaluation

In [20]:
r2_score(y_test, grid_search.predict(X_test))

0.428532454726153

In [21]:
submission_data = pd.read_csv(path2)

submission_predictions = pd.DataFrame({
    'Id': submission_data['Id'],
    'SalePrice': grid_search.predict(submission_data)
})

submission_predictions.to_csv('sub_1.csv', index=False)